Сначала создаем и обучаем модель классификации.

In [1]:
import gdown
import zipfile
import os
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Subset
import cv2

Загружаем датасет LogoDet-3K из архива. Датасет в виде архива сохранен на моем Google Drive.

In [2]:
url = f'https://drive.google.com/file/d/1J-8R6iduBKUd1vRBKr6_eWcFRmzFPeRj/view?usp=drive_link'
output = 'logodet-3k.zip'

gdown.download(url, output, quiet=False)

with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall('data/logos')

os.remove(output)

Downloading...
From (original): https://drive.google.com/uc?id=1J-8R6iduBKUd1vRBKr6_eWcFRmzFPeRj
From (redirected): https://drive.google.com/uc?id=1J-8R6iduBKUd1vRBKr6_eWcFRmzFPeRj&confirm=t&uuid=68d69f5b-27d4-4c1c-9793-a12b13950566
To: /home/grakoka/GRAKOKA/21school/logo_detection-main/logodet-3k.zip
100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 3.08G/3.08G [04:30<00:00, 11.4MB/s]


In [3]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

In [4]:
data_dir = 'data/logos/LogoDet-3K'

dataset = datasets.ImageFolder(data_dir, transform=transform)

Разделяем датасет на обучающую и тестовую выборки

In [5]:
train_indices, val_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.targets,
    random_state=42
)

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

Создаем DataLoader

In [6]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

Обучаем модель классификации. Я использовал модель ResNet.

In [7]:
# Преобразуем изображение для классификации
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [8]:
model_classifier = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_classifier.fc = nn.Linear(model_classifier.fc.in_features, len(dataset.classes))

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_classifier.parameters(), lr=0.001)

In [10]:
num_epochs = 10
for epoch in range(num_epochs):
    model_classifier.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_classifier(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

Epoch 1/10, Loss: 1.2935854837658904
Epoch 2/10, Loss: 0.9251829961975736
Epoch 3/10, Loss: 0.6378559337388123
Epoch 4/10, Loss: 0.4222324124927963
Epoch 5/10, Loss: 0.27652570524162823
Epoch 6/10, Loss: 0.199776811044543
Epoch 7/10, Loss: 0.15695501322061906
Epoch 8/10, Loss: 0.13527353441027262
Epoch 9/10, Loss: 0.11778009291588058
Epoch 10/10, Loss: 0.10048062946823906


Сохраняем модель

In [12]:
torch.save(model_classifier.state_dict(), 'models/logo_classifier.pth')

Используем обученную модель для получения предсказания

In [20]:
model_classifier = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_classifier.fc = nn.Linear(model_classifier.fc.in_features, 9)  
model_classifier.load_state_dict(torch.load('models/logo_classifier.pth'))
model_classifier.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [21]:
# Преобразуем изображение для классификации
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [22]:
def classify_logo(image):
    image = preprocess(image).unsqueeze(0)
    with torch.no_grad():
        prediction = model_classifier(image)
    return torch.argmax(prediction)

In [28]:
import cv2
image = cv2.imread('data/logos/LogoDet-3K/Medical/Calpol/0.jpg')
result = classify_logo(image)
print(f"Classification result: {result}")

Classification result: 6


In [29]:
model_classifier = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_classifier.fc = nn.Linear(model_classifier.fc.in_features, 9) 
model_classifier.load_state_dict(torch.load('models/logo_classifier.pth'))
model_classifier.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con